# 00 — Getting started

Exploration only (see `docs/deltaprinciples.md` §2). Stable logic moves to `src/usse/jobs/` with tests.

The Spark session gets all config (Delta, S3 storage, metastore) from `spark-defaults.conf`, same as jobs.

In [ ]:
from usse.settings import load_settings
from usse.spark import get_spark

settings = load_settings()
spark = get_spark("nb-getting-started")
spark.sql("SHOW DATABASES").show()

## A Delta table, its versions, and time travel

In [ ]:
table = f"{settings.schema('slv')}._nb_demo"
spark.sql(f"DROP TABLE IF EXISTS {table}")
spark.sql(f"CREATE TABLE {table} (post_id STRING, likes INT) USING DELTA")                 # v0
spark.sql(f"INSERT INTO {table} VALUES ('st:1', 3), ('st:2', 5)")                            # v1
spark.sql(f"UPDATE {table} SET likes = 10 WHERE post_id = 'st:1'")                           # v2
spark.sql(f"DESCRIBE HISTORY {table}").select("version", "operation").show()

In [ ]:
spark.sql(f"SELECT * FROM {table} VERSION AS OF 1").show()   # before the UPDATE
spark.sql(f"SELECT * FROM {table}").show()                    # current

In [ ]:
spark.sql(f"DROP TABLE {table}")